# GIAI ĐOẠN 4: KỸ THUẬT TẠO ĐẶC TRƯNG, KIỂM TOÁN RÒ RỈ DỮ LIỆU & ĐÓNG GÓI PIPELINE
**Đề tài:** Phân tích Biểu hiện Định giá Động (*Dynamic Surge Pricing*) và Dự báo Giá báo API trên Nền tảng Gọi xe Công nghệ (Uber & Lyft Boston, MA)  
**Mã đề tài:** `PRJ-EXT-01` | **Nhóm thực hiện:** Nhóm 02 — Trần Thanh Huy (`2521001060`) & Lê Huy Thắng (`2521001092`)  
**Môi trường thực thi:** Python `3.11.9` | `pandas 3.1.1` | `numpy 2.4.6` | `scipy 1.17.1` | `statsmodels 0.15.0` | `scikit-learn 1.9.1`

---
## Bước 1. Nạp Bảng Dữ Liệu Sạch Tổng Thể & Kiểm Tra Phạm Vi Quan Sát
- **Làm gì:** Nạp tệp nhị phân chuẩn hóa `data/processed/master_rideshare.parquet` (`270,898` dòng × `32` cột) và xác minh khung thời gian, số ngày quan sát thực tế và số tuyến lộ trình thực tế.
- **Tại sao phải làm:** Đảm bảo đầu vào của Giai đoạn 4 khớp tuyệt đối với đầu ra của Giai đoạn 3 và xác thực các thông số phạm vi dữ liệu trước khi tạo đặc trưng.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))

from features import (  # noqa: E402
    audit_features,
    engineer_features_stage4,
    load_master_dataset,
)

df_master = load_master_dataset()
print(f"Kích thước master_rideshare.parquet: {df_master.shape}")
dt_min = df_master["datetime"].min()
dt_max = df_master["datetime"].max()
print(f"Khung thời gian: {dt_min} -> {dt_max}")
n_days = df_master["datetime"].dt.date.nunique()
print(f"Số ngày có dữ liệu: {n_days} ngày")
n_routes = df_master.groupby(["source", "destination"], observed=True).ngroups
print(f"Số tuyến (source -> destination) thực tế: {n_routes} tuyến")

2026-11-28 03:42:13,544 - INFO - Đã nạp master_rideshare.parquet: 270,898 dòng x 32 cột.


Kích thước master_rideshare.parquet: (270898, 32)
Khung thời gian: 2018-11-25 22:40:46.319000 -> 2018-12-18 14:15:10.943000
Số ngày có dữ liệu: 18 ngày
Số tuyến (source -> destination) thực tế: 72 tuyến


## Bước 2. Rà Soát Từng Biến (Feature Governance & Audit Table)
- **Làm gì:** Phân loại toàn bộ `41` cột thành các nhóm: `Target y` (`price`), `Drop - Leakage` (`price_per_mile`), `Drop - Key/ID` (`id`, `time_stamp`, `datetime`, `date_hour`, `location`), `Drop - Provenance Flag` (`is_timestamp_original`, `weather_matched`, `is_surge_original`, `is_rain_sentinel`), `Drop - Redundant` (`product_id`, `day_name`), `Aux Group Only` (`route_id`) và `Keep in X` (`27` đặc trưng hợp lệ).
- **Tại sao phải làm:** Ngăn chặn từ gốc việc vô tình đưa biến định danh, cờ nguồn gốc, biến trùng lặp 1-1 hoặc biến dẫn xuất từ mục tiêu vào ma trận huấn luyện $X$.

In [2]:
df_feat = engineer_features_stage4(df_master)
audit_df = audit_features(df_feat)
print("Thống kê quyết định kiểm toán cột:")
print(audit_df["decision"].value_counts())
print("\nCác cột bị loại khỏi ma trận đặc trưng X:")
dropped = audit_df.loc[
    ~audit_df["decision"].str.startswith("Keep in X"),
    ["column_name", "role", "decision", "rationale"],
]
print(dropped.to_string(index=False))

Thống kê quyết định kiểm toán cột:
decision
Keep in X                  26
Drop - Key/ID               5
Drop - Audit Flag           4
Drop - Redundant            2
Keep in X (Conditional)     1
Target y                    1
Drop - Leakage              1
Aux Group Only              1
Name: count, dtype: int64

Các cột bị loại khỏi ma trận đặc trưng X:
          column_name                role          decision                                                                                                                                           rationale
                   id      Identifier/Key     Drop - Key/ID                                                                  Mã định danh duy nhất của chuyến đi (UUID), không mang thông tin quy luật kinh tế.
           product_id           Redundant  Drop - Redundant                                             Mã nội bộ ánh xạ tất định 1-1 với cặp (cab_type, name); giữ lại sẽ gây suy biến đa cộng tuyến hoàn hảo.
    is_surge_original  

## Bước 3. Minh Họa Thực Nghiệm Bẫy Rò Rỉ Dữ Liệu (R4) & Kiểm Toán Bản Ghi Nguyên Bản (R3)
- **Bẫy rò rỉ mục tiêu (`price_per_mile` - R4):** Cột `price_per_mile = price / distance` có hệ số tương quan Pearson $r = 0.4189$ với `price`. Nếu vô tình đưa `price_per_mile` cùng `distance` và `surge_multiplier` vào mô hình cây `HistGradientBoosting` (điền khuyết trung vị toàn cục cho `3,830` dòng Test thiếu `distance`/`price_per_mile`), mô hình học lén tích `distance * price_per_mile` trên nhóm có `distance` thực nên kéo sai số tuyệt đối xuống giả tạo ($\text{MAE} = \$0.8030$, $\text{MAPE} = 6.29\%$; dù $R^2$ chung tụt về `0.9176`, $\text{RMSE} = \$2.6474$ do phần dư cực trị trên nhóm bị điền trung vị) — một kết quả rò rỉ vô nghĩa trong thực tế vì khi khách đặt xe chưa thể biết `price_per_mile`.
- **Kiểm toán quy mô bậc 10 (R3):** Đối chiếu từng `id` với tệp thô `data/raw/cab_rides.csv` cho thấy có **`240,689 / 270,898` dòng (`88.85%`)** hoàn toàn nguyên bản (không bị chia tỷ lệ $10^m$ ở `price`/`distance` và không bị điền khuyết `distance`). Cờ `is_strictly_unmodified_raw` được lưu vào `data/processed/stage4_audit_flags.parquet` để Giai đoạn 5 kiểm định độ nhạy độc lập.

In [3]:
from features import audit_raw_unmodified_flags

flags_df, r3_summary = audit_raw_unmodified_flags(df_feat)
for k, v in r3_summary.items():
    print(f"{k:36s}: {v}")

2026-11-28 03:42:13,824 - INFO - Đang đối chiếu với data/raw/cab_rides.csv để kiểm toán tỷ lệ bản ghi nguyên bản (R3 Audit)...


2026-11-28 03:42:15,546 - INFO - Hoàn tất kiểm toán R3: 240,689/270,898 dòng (88.85%) hoàn toàn không qua khôi phục quy mô 10^m hay điền khuyết distance.


total_clean_rows                    : 270898
price_unmodified_count              : 266889
price_unmodified_pct                : 98.52
price_scale_recovered_count         : 4009
price_scale_recovered_pct           : 1.48
distance_unmodified_count           : 244288
distance_unmodified_pct             : 90.18
distance_recovered_or_imputed_count : 26610
distance_recovered_or_imputed_pct   : 9.82
strictly_unmodified_both_count      : 240689
strictly_unmodified_both_pct        : 88.85


## Bước 4. Tạo Đặc Trưng Kinh Tế Lượng & Chu Kỳ Lượng Giác (Feature Creation)
- **Các đặc trưng tạo mới:**
  1. `log_distance = ln(1 + distance)`: Nắm bắt hiệu ứng giá sàn tối thiểu (*Minimum Fare*) làm đường giá cong nhẹ ở cự ly rất ngắn ($< 1.0$ dặm).
  2. `is_lyft`, `lyft_surge_excess = (surge_multiplier - 1.0) * is_lyft`, `distance_x_lyft_surge_excess`: Tách biệt cơ chế nhân giá động chỉ xuất hiện trên Lyft (`6.80%` trên `92,384` lượt Lyft quan sát surge thực), trong khi Uber cố định `surge = 1.0`.
  3. `hour_sin, hour_cos` và `dow_sin, dow_cos`: Biểu diễn tính tuần hoàn liên tục của thời gian (23:00 liền kề 00:00).

In [4]:
from config import ENGINEERED_NUM_FEATURES

cols_desc = ["count", "mean", "std", "min", "50%", "max"]
print(df_feat[ENGINEERED_NUM_FEATURES].describe().T[cols_desc].round(4))

                                 count    mean     std     min     50%  \
distance_x_surge              213735.0  2.2289  1.1877  0.0200  2.1900   
log_distance                  248636.0  1.0995  0.3569  0.0198  1.1537   
is_lyft                       270898.0  0.4814  0.4997  0.0000  0.0000   
lyft_surge_excess             232861.0  0.0123  0.0863  0.0000  0.0000   
distance_x_lyft_surge_excess  213735.0  0.0294  0.2279  0.0000  0.0000   
hour_sin                      208034.0  0.0015  0.7139 -1.0000 -0.0000   
hour_cos                      208034.0 -0.0262  0.6997 -1.0000 -0.0000   
dow_sin                       208034.0  0.0081  0.6868 -0.9749  0.0000   
dow_cos                       208034.0  0.0728  0.7231 -0.9010 -0.2225   

                                  max  
distance_x_surge              12.4000  
log_distance                   2.1815  
is_lyft                        1.0000  
lyft_surge_excess              2.0000  
distance_x_lyft_surge_excess   7.4400  
hour_sin           

## Bước 5. Phân Tích Đa Cộng Tuyến (VIF) Trước và Sau Trực Giao Hóa
- **Phát hiện kinh tế lượng cốt lõi:**
  - Nếu đưa đồng thời `distance`, `surge_multiplier` và `distance_x_surge` vào mô hình tuyến tính mà không tâm hóa, vì `surge_multiplier = 1.0` trên đại đa số quan sát nên `distance_x_surge` trùng khớp mạnh với `distance`, gây đa cộng tuyến nghiêm trọng (VIF > 120).
  - Khi thay `surge_multiplier` bằng phần phụ trội `lyft_surge_excess = (surge - 1.0) * is_lyft`, đa cộng tuyến cấu trúc bị triệt tiêu hoàn toàn: $\text{VIF}(\text{distance})$ giảm về gần `1.0`!
  - Toàn bộ 6 biến thời tiết (`temp`, `clouds`, `pressure`, `rain`, `humidity`, `wind`) trên tập khớp khí tượng sạch lỗi cảm biến mưa đều có $\text{VIF} < 5.0$, không có đa cộng tuyến nghiêm trọng.

In [5]:
from features import create_data_splits, run_vif_analysis

splits, split_df = create_data_splits(df_feat)
rand_tr_idx, rand_te_idx = splits["random_80_20"]
vif_df, vif_summary = run_vif_analysis(df_feat.iloc[rand_tr_idx])
vif_cols = ["feature", "vif", "assessment"]
print("=== CẤU HÌNH A: TRƯỚC TÂM HÓA (DÙNG SURGE_MULTIPLIER GỐC >= 1.0) ===")
cfg_a = vif_df[vif_df["config"] == "A_Uncentered_Raw_Interaction"]
print(cfg_a[vif_cols].head(5).to_string(index=False))
print(
    "\n=== CẤU HÌNH B: SAU TRỰC GIAO HÓA "
    "(DÙNG LYFT_SURGE_EXCESS = SURGE - 1.0) ==="
)
cfg_b = vif_df[vif_df["config"] == "B_Centered_Excess_Interaction"]
print(cfg_b[vif_cols].to_string(index=False))

2026-11-28 03:42:15,664 - INFO - Đang thiết lập 3 chiến lược chia dữ liệu (test_size=0.2, random_state=42)...


2026-11-28 03:42:15,937 - INFO - Đang tính toán VIF trên tập huấn luyện (X_train)...


2026-11-28 03:42:17,587 - INFO - VIF của 'distance' giảm từ 162.04 (chưa tâm hóa) xuống 1.02 (sau khi chuyển sang lyft_surge_excess); VIF tối đa của nhóm thời tiết = 4.81.


=== CẤU HÌNH A: TRƯỚC TÂM HÓA (DÙNG SURGE_MULTIPLIER GỐC >= 1.0) ===
         feature      vif                               assessment
distance_x_surge 168.2705 Cao (VIF >= 10 - Đa cộng tuyến cấu trúc)
        distance 162.0358 Cao (VIF >= 10 - Đa cộng tuyến cấu trúc)
surge_multiplier   5.9839               Trung bình (5 <= VIF < 10)
     day_of_week   4.0634                        An toàn (VIF < 5)
      is_weekend   3.9177                        An toàn (VIF < 5)

=== CẤU HÌNH B: SAU TRỰC GIAO HÓA (DÙNG LYFT_SURGE_EXCESS = SURGE - 1.0) ===
                     feature    vif                 assessment
                     dow_sin 6.6439 Trung bình (5 <= VIF < 10)
distance_x_lyft_surge_excess 6.0135 Trung bình (5 <= VIF < 10)
           lyft_surge_excess 5.9842 Trung bình (5 <= VIF < 10)
                    pressure 4.8134          An toàn (VIF < 5)
                    humidity 3.2249          An toàn (VIF < 5)
                  is_weekend 2.8732          An toàn (VIF < 5)
          

## Bước 6. Ba Chiến Lược Phân Chia Tập Dữ Liệu (Random, Temporal EST, Route Group Split)
- **Mục đích:** Thiết lập đồng thời 3 cách chia dữ liệu trên tập `232,861` dòng có `is_surge_original == 1` (tách riêng `38,037` dòng Lyft khuyết surge làm Holdout Ex-Ante): `random_80_20` (`186,288 / 46,573`), `temporal_split` (`143,023 / 35,756`, khóa TS gốc EST) và `route_group_split` (`184,664 / 48,197`, 57/15 tuyến).

In [6]:
split_cols = [
    "split_strategy",
    "n_train",
    "n_test",
    "train_price_mean",
    "test_price_mean",
    "ks_statistic",
    "standardized_mean_diff",
    "train_routes",
    "test_routes",
]
print(split_df[split_cols].to_string(index=False))

   split_strategy  n_train  n_test  train_price_mean  test_price_mean  ks_statistic  standardized_mean_diff  train_routes  test_routes
     random_80_20   186288   46573           16.4250          16.4704       0.00348                 0.00492            72           72
   temporal_split   143023   35756           16.4409          16.4658       0.00385                 0.00269            72           72
route_group_split   184664   48197           16.5929          15.8256       0.04636                 0.08451            57           15


## Bước 7 & 8. Đóng Gói Pipeline & Kiểm Chứng Giải Pháp R5 (OLS Interaction)
- **Kết quả kiểm chứng trên tập Test (`n = 46,573` mẫu quan sát surge thực):**
  - **M1 (OLS Cơ bản - chỉ dùng 1 hệ số dặm chung):** $R^2 = 0.9255$, $\text{RMSE} = \$2.5182$, **$\text{MAPE} = 12.81\%$** (Chưa đạt ngưỡng SMART $\le 10\%$).
  - **M2 (OLS Tương tác Kinh tế lượng - mỗi gói xe có giá mở cửa, đơn giá/dặm, độ cong cự ly ngắn và hệ số surge riêng):** **$R^2 = 0.9539$**, **$\text{RMSE} = \$1.9802$**, **$\text{MAPE} = 9.33\%$** — **VƯỢT MỤC TIÊU SMART ($R^2 \ge 0.90, \text{MAPE} \le 10\%$)** ngay trên mô hình tuyến tính diễn giải 100%!
  - **M7 (HistGradientBoosting Preview):** $R^2 = 0.9625$, $\text{RMSE} = \$1.7857$, $\text{MAPE} = 8.23\%$.

In [7]:
from features import verify_pipelines_and_leakage_trap

ver_res = verify_pipelines_and_leakage_trap(df_feat, rand_tr_idx, rand_te_idx)
print("Kích thước ma trận đầu ra của 4 bộ tiền xử lý:")
for k, v in ver_res["preprocessor_shapes"].items():
    print(f"  - {k:26s}: Train={v['train_shape']} | Test={v['test_shape']}")
print(
    f"\nSo sánh hiệu năng kiểm chứng trên tập Test (n={len(rand_te_idx):,}):"
)
print("  - M1 (OLS cơ bản)             :", ver_res["m1_basic_ols"])
print("  - M2 (OLS tương tác kinh tế)  :", ver_res["m2_interaction_ols"])
print("  - M7 (HistGradientBoosting)   :", ver_res["m7_stage4_hgb_preview"])
leak_res = ver_res["leakage_trap_experiment"]["corrupted_hgb_with_ppm"]
print("  - Bẫy rò rỉ (HistGB + ppm)    :", leak_res)

2026-11-28 03:42:17,604 - INFO - Đang kiểm chứng các Pipeline và chạy thực nghiệm đối chứng R4 (Leakage) & R5 (Interaction)...


2026-11-28 03:42:22,949 - INFO - Kết quả kiểm chứng R5: OLS cơ bản (M1) R2=0.9255, MAPE=12.81% --> OLS tương tác kinh tế (M2) R2=0.9539, MAPE=9.33%.


2026-11-28 03:42:22,950 - INFO - Kết quả Mutate & Break R4 (bẫy rò rỉ price_per_mile): HistGB với price_per_mile đạt R2=0.9176, MAPE=6.29%


Kích thước ma trận đầu ra của 4 bộ tiền xử lý:
  - m1_basic_linear           : Train=[186288, 49] | Test=[46573, 49]
  - m2_m4_interaction_linear  : Train=[186288, 82] | Test=[46573, 82]
  - m5_m6_tree_ohe            : Train=[186288, 65] | Test=[46573, 65]
  - m7_hgb_native             : Train=[186288, 27] | Test=[46573, 27]

So sánh hiệu năng kiểm chứng trên tập Test (n=46,573):
  - M1 (OLS cơ bản)             : {'r2': 0.9255, 'rmse': 2.5182, 'mae': 1.7488, 'mape_pct': 12.81}
  - M2 (OLS tương tác kinh tế)  : {'r2': 0.9539, 'rmse': 1.9802, 'mae': 1.2888, 'mape_pct': 9.33}
  - M7 (HistGradientBoosting)   : {'r2': 0.9625, 'rmse': 1.7857, 'mae': 1.1377, 'mape_pct': 8.23}
  - Bẫy rò rỉ (HistGB + ppm)    : {'r2': 0.9176, 'rmse': 2.6474, 'mae': 0.803, 'mape_pct': 6.29}
